##  Inventory Management - Aligned Pipeline

This notebook reflects the final aligned pipeline for inventory forecasting.

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
import joblib
import os

data_path = "../data/Processed/Inventory_management_cleaned.csv"
if not os.path.exists(data_path):
    data_path = "c:/Users/LENOVO/Downloads/Zidio project/RetailPulse/data/Processed/Inventory_management_cleaned.csv"

df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store_ID_enc', 'Product_ID_enc', 'Date'])

# Engineering
df['target'] = df.groupby(['Store_ID_enc', 'Product_ID_enc'])['Units Sold'].shift(-1)
for lag in [1, 7, 14, 30]:
    df[f'lag_{lag}'] = df.groupby(['Store_ID_enc', 'Product_ID_enc'])['Units Sold'].shift(lag)

df['ema_7'] = df.groupby(['Store_ID_enc', 'Product_ID_enc'])['Units Sold'].shift(1).transform(lambda x: x.ewm(span=7).mean())
df['ema_30'] = df.groupby(['Store_ID_enc', 'Product_ID_enc'])['Units Sold'].shift(1).transform(lambda x: x.ewm(span=30).mean())

for window in [7, 30]:
    df[f'rolling_mean_{window}'] = df.groupby(['Store_ID_enc', 'Product_ID_enc'])['Units Sold'].shift(1).rolling(window).mean()

df['month_sin'] = np.sin(2 * np.pi * df['Date'].dt.month / 12)
df['month_cos'] = np.cos(2 * np.pi * df['Date'].dt.month / 12)
df['dow_sin'] = np.sin(2 * np.pi * df['Date'].dt.dayofweek / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['Date'].dt.dayofweek / 7)

df = df.dropna(subset=['target']).dropna()

features = [
    'lag_1', 'lag_7', 'lag_14', 'lag_30',
    'ema_7', 'ema_30',
    'rolling_mean_7', 'rolling_mean_30',
    'Price', 'Discount', 'Holiday/Promotion',
    'month_sin', 'month_cos',
    'dow_sin', 'dow_cos',
    'Store_ID_enc', 'Product_ID_enc'
]

X = df[features]
y = df['target']

model = XGBRegressor(objective='reg:tweedie', n_estimators=500)
model.fit(X, y)

joblib.dump(features, '../models/inventory_features.pkl')
joblib.dump(model, '../models/inventory_model.pkl')
print("Model and Features saved.")